<a href="https://colab.research.google.com/github/dguemo/si3003-workshop-rag-dguemo/blob/main/notebooks/03_rag_workshop_papers_nvidia_nim_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Workshop — RAG sobre papers académicos con LangChain y NVIDIA NIM — versión de David (dguemo)

En este workshop construirás un sistema RAG completo alimentado con dos papers fundamentales:

- **Attention Is All You Need** — Vaswani et al. (2017)
- **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks** — Lewis et al. (2020)

El notebook descarga los PDFs desde arXiv, extrae su texto, crea embeddings y responde preguntas usando únicamente el contenido indexado.

**Stack:** MarkItDown · LangChain · NVIDIA NIM · Qdrant

| Parte | Tarea |
|---|---|
| 0 | Instalar y configurar Colab |
| 1 | Descargar y convertir PDFs con MarkItDown |
| 2 | Crear `Document` y aplicar chunking |
| 3 | Generar embeddings NVIDIA e indexar en Qdrant |
| 4 | Construir el pipeline RAG con LCEL |
| 5 | Evaluar mediante preguntas con *ground truth* |

**Modelos:**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3-super-120b-a12b`


**Tareas resueltas en esta copia:** B1 (tabla de evaluación tras el Bonus), B2 (experimentos), B3 (preguntas trampa), B4 (español vs inglés), B5 (quiz propio) y R1 (reflexión). Cada celda de tarea imprime una tabla Markdown lista para `RESPUESTAS.md`.

In [1]:
# Dependencias para Google Colab
%pip install -qU "markitdown[pdf]" langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv requests


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.4/62.4 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.5/170.5 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 574.7/574.7 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.3/89.3 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 471.2/471.2 kB 26.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.5/136.5 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.2/16.2 MB 39.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 98.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.1

In [2]:
import os
import getpass
from pathlib import Path
from typing import List

import requests
from dotenv import load_dotenv

load_dotenv()

try:
    from google.colab import userdata
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except (ImportError, KeyError):
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY
GENERATION_MODEL = "nvidia/nemotron-3-super-120b-a12b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

print("✅ Configuración NVIDIA NIM lista")

# ── Elegir un modelo generativo disponible para esta cuenta ──────────────
# nemotron-3-super salió del catálogo gratuito (404) y varios modelos viejos ya están
# retirados (410 Gone). Se prueban modelos vigentes del catálogo de build.nvidia.com y,
# si ninguno responde, los modelos de chat que la API lista para esta cuenta.
from langchain_nvidia_ai_endpoints import ChatNVIDIA

CANDIDATE_MODELS = [
    GENERATION_MODEL,                             # nvidia/nemotron-3-super-120b-a12b (el del curso)
    "nvidia/nemotron-3.5-lightning-30b-a3b",
    "google/gemma-4-31b-it",
    "nvidia/nemotron-3-ultra-550b-a55b",
    "openai/gpt-oss-20b",
    "deepseek-ai/deepseek-v4.1-flash",
    "z-ai/glm-5.3-flash",
]

def discovered_chat_models():
    try:
        models = ChatNVIDIA.get_available_models()
        return [m.id for m in models if getattr(m, "model_type", "chat") in (None, "chat", "vlm", "qa")]
    except Exception as error:
        print(f"⚠️ No se pudo listar modelos: {error}")
        return []

def works(model):
    try:
        out = ChatNVIDIA(model=model, max_completion_tokens=300).with_thinking_mode(enabled=False).invoke("Responde solo con la palabra: listo")
        if not (out.content or "").strip():
            print(f"⚠️ {model} respondió vacío (gasta los tokens razonando); se descarta")
            return False
        return True
    except Exception as error:
        print(f"⚠️ {model} no disponible: {str(error).splitlines()[0][:120]}")
        return False

def pick_generation_model(candidates, max_extra=12):
    for model in candidates:
        if works(model):
            return model
    extra = [m for m in discovered_chat_models() if m not in candidates][:max_extra]
    print(f"ℹ️ Probando {len(extra)} modelos listados por la API...")
    for model in extra:
        if works(model):
            return model
    raise RuntimeError("Ningún modelo generativo respondió: revisa tu key en build.nvidia.com")

GENERATION_MODEL = pick_generation_model(CANDIDATE_MODELS)
print(f"✅ Modelo generativo: {GENERATION_MODEL}")


✅ Configuración NVIDIA NIM lista
⚠️ nvidia/nemotron-3-super-120b-a12b no disponible: [404] Not Found
✅ Modelo generativo: nvidia/nemotron-3.5-lightning-30b-a3b


---
## Parte 1 — Descargar y convertir los PDFs

La celda descarga los artículos únicamente si todavía no existen en `papers/`. Después, MarkItDown convierte cada PDF a texto conservando buena parte de su estructura.

> Si una sesión de Colab se reinicia, los archivos temporales se pierden y se descargarán nuevamente.


In [3]:
from markitdown import MarkItDown

DOCS_DIR = Path("papers")
DOCS_DIR.mkdir(parents=True, exist_ok=True)

PAPERS = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_paper.pdf": "https://arxiv.org/pdf/2005.11401",
}


def download_file(url: str, destination: Path) -> None:
    if destination.exists() and destination.stat().st_size > 10_000:
        print(f"↪ Ya existe: {destination.name}")
        return

    try:
        response = requests.get(url, timeout=90)
        response.raise_for_status()
        if not response.content.startswith(b"%PDF"):
            raise ValueError(f"La descarga de {url} no produjo un PDF válido")
    except Exception as error:
        # Respaldo: copia de los PDFs en el repositorio del curso
        backup = f"https://raw.githubusercontent.com/EAFIT-IA/si3003-artificial-intelligence/main/notebooks/lecture10/papers/{destination.name}"
        print(f"⚠️ arXiv falló ({error}); probando {backup}")
        response = requests.get(backup, timeout=90)
        response.raise_for_status()
        if not response.content.startswith(b"%PDF"):
            raise ValueError(f"Tampoco se obtuvo un PDF válido para {destination.name}")
    destination.write_bytes(response.content)
    print(f"⬇ Descargado: {destination.name} ({destination.stat().st_size / 1e6:.1f} MB)")


for filename, url in PAPERS.items():
    download_file(url, DOCS_DIR / filename)

md_converter = MarkItDown(enable_plugins=False)


def load_pdf(path: str | Path) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Archivo no encontrado: {path}")
    result = md_converter.convert(str(path))
    text = result.text_content
    if not text or not text.strip():
        raise ValueError(f"No se pudo extraer texto de {path.name}")
    return text


attention_text = load_pdf(DOCS_DIR / "attention_paper.pdf")
rag_text = load_pdf(DOCS_DIR / "rag_paper.pdf")

print(f"✅ attention_paper.pdf → {len(attention_text):,} caracteres")
print(f"✅ rag_paper.pdf       → {len(rag_text):,} caracteres")
print(f"\nInicio del paper de Attention:\n{attention_text[:300]}...")


⬇ Descargado: attention_paper.pdf (2.2 MB)
⬇ Descargado: rag_paper.pdf (0.9 MB)
✅ attention_paper.pdf → 40,174 caracteres
✅ rag_paper.pdf       → 69,121 caracteres

Inicio del paper de Attention:
Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor
scholarlyworks.
Attention Is All You Need
AshishVaswani∗ NoamShazeer∗ NikiParmar∗ JakobUszkoreit∗
GoogleBrain GoogleBrain GoogleResearch GoogleResearch
avaswani@goog...


---
## Parte 2 — Convertir a `Document` y aplicar chunking

LangChain utiliza objetos `Document` con dos campos principales:

- `page_content`: contenido textual del fragmento.
- `metadata`: título, archivo de origen e índice del fragmento.

Usaremos fragmentos de 900 caracteres con solapamiento de 120. Es un punto de partida didáctico; no es un valor universal.


In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


def pdf_to_documents(text: str, title: str, source: str) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=900,
        chunk_overlap=120,
        separators=["\n\n", "\n", ".", " "],
    )
    documents = []
    for index, chunk in enumerate(splitter.split_text(text)):
        if len(chunk.strip()) < 30:
            continue
        documents.append(Document(
            page_content=chunk,
            metadata={"title": title, "source": source, "chunk_index": index},
        ))
    return documents


attention_docs = pdf_to_documents(
    attention_text,
    title="Attention Is All You Need",
    source="attention_paper.pdf",
)
rag_docs = pdf_to_documents(
    rag_text,
    title="Retrieval-Augmented Generation",
    source="rag_paper.pdf",
)
all_docs = attention_docs + rag_docs

print(f"Attention → {len(attention_docs)} chunks")
print(f"RAG       → {len(rag_docs)} chunks")
print(f"Total     → {len(all_docs)} chunks")
print("\nEjemplo de metadatos:", all_docs[0].metadata)


Attention → 55 chunks
RAG       → 95 chunks
Total     → 150 chunks

Ejemplo de metadatos: {'title': 'Attention Is All You Need', 'source': 'attention_paper.pdf', 'chunk_index': 0}


---
## Parte 3 — Embeddings NVIDIA e índice Qdrant

`NVIDIAEmbeddings` distingue automáticamente entre embeddings de documentos y consultas mediante `embed_documents()` y `embed_query()`. La dimensión se obtiene directamente del modelo para evitar valores codificados manualmente.

Qdrant se ejecuta en memoria; al reiniciar el runtime se reconstruye el índice.


In [5]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "academic_papers_nvidia"

embeddings = NVIDIAEmbeddings(
    model=EMBEDDING_MODEL,
    truncate="END",
)
EMBEDDING_DIM = len(embeddings.embed_query("dimension probe"))

qdrant_client = QdrantClient(":memory:")
qdrant_client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
ids = vectorstore.add_documents(all_docs)

print(f"✅ {len(ids)} chunks indexados en Qdrant")
print(f"✅ Dimensión de embeddings: {EMBEDDING_DIM}")


/usr/local/lib/python3.13/dist-packages/langchain_nvidia_ai_endpoints/_common.py:269: UserWarning: Found nvidia/nemotron-3-embed-1b in available_models, but type is unknown and inference may fail.
  warnings.warn(


✅ 150 chunks indexados en Qdrant
✅ Dimensión de embeddings: 2048


---
## Parte 4 — Pipeline RAG con LCEL

El pipeline conecta recuperación, construcción del prompt, generación con Nemotron y extracción del texto final. El prompt obliga al modelo a reconocer cuando el contexto recuperado no contiene la respuesta.


In [6]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Modo razonamiento apagado y margen de tokens para evitar respuestas vacías
llm = ChatNVIDIA(
    model=GENERATION_MODEL,
    temperature=0.2,
    top_p=0.95,
    max_completion_tokens=2048,
).with_thinking_mode(enabled=False)

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 5, "score_threshold": 0.25},
)


def format_docs(documents: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[{doc.metadata.get('title', 'Untitled')} | {doc.metadata.get('source', '')}]"
        f"\n{doc.page_content}"
        for doc in documents
    )


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer using only the supplied academic-paper excerpts. "
        "If the excerpts do not contain enough evidence, say so explicitly. "
        "Answer in the same language as the question and be concise. "
        "Mention the paper title supporting the answer.",
    ),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

test_question = "What is the Transformer architecture?"
test_answer = rag_chain.invoke(test_question)
print("Pregunta:", test_question)
print("Respuesta:", test_answer)


Pregunta: What is the Transformer architecture?
Respuesta: The Transformer architecture is a sequence transduction model based solely on attention mechanisms, dispensing with recurrence and convolutions entirely. It uses stacked self-attention and point-wise, fully connected layers for both the encoder and decoder. Each encoder layer contains a multi-head self-attention mechanism and a position-wise feed-forward network, with residual connections and layer normalization. The model achieves state-of-the-art performance in machine translation tasks while being more parallelizable and requiring significantly less training time compared to previous architectures.


### Utilidades para las tareas

`with_retry` maneja el rate limit de NVIDIA, `grade` hace un chequeo automático por palabras clave contra el ground truth (yo reviso a mano los casos «parcial»), y `retrieved` lista los chunks con su **coseno crudo**.

> Hallazgo: `langchain-qdrant` compara `score_threshold` contra `(coseno + 1) / 2`, no contra el coseno. Por eso `score_threshold=0.25` equivale a coseno ≥ −0.5 y casi nunca descarta nada. La celda de abajo lo comprueba.

In [7]:
import re, time

def with_retry(fn, *args, tries=6, wait=20, **kwargs):
    for attempt in range(1, tries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as error:
            if any(s in str(error) for s in ("429", "Too Many", "rate limit", "Rate limit")) and attempt < tries:
                print(f"⏳ Rate limit; reintento en {wait}s"); time.sleep(wait); continue
            raise

def to_relevance(cos: float) -> float:
    return (cos + 1) / 2

def cell(text, n=200):
    text = re.sub(r"\s+", " ", str(text)).replace("|", "\\|").strip()
    return text if len(text) <= n else text[:n - 1] + "…"

REFUSAL = ("do not contain", "don't contain", "does not contain", "not enough evidence", "no evidence",
           "not mentioned", "not provided", "cannot answer", "can't answer", "insufficient",
           "no contienen", "no contiene", "no hay evidencia", "no se menciona", "no encuentro",
           "do not provide", "does not provide", "not specified", "no information")

def is_refusal(answer: str) -> bool:
    return any(s in answer.lower() for s in REFUSAL)

def grade(answer: str, groups) -> str:
    """groups: lista de grupos de palabras clave (OR dentro del grupo, AND entre grupos)."""
    low = answer.lower()
    hits = sum(any(w.lower() in low for w in g) for g in groups)
    return "sí" if hits == len(groups) else ("parcial" if hits else "no")

def retrieved(store, question, k=5):
    """[(title, source, chunk_index, coseno)] sin aplicar umbral."""
    return [(d.metadata["title"], d.metadata["source"], d.metadata["chunk_index"], s)
            for d, s in store.similarity_search_with_score(question, k=k)]

def fmt_retrieved(items):
    return "; ".join(f"{src.split('_')[0]}#{idx} ({s:.2f})" for _, src, idx, s in items)

# Comprobación de la escala del umbral
_raw = vectorstore.similarity_search_with_score(test_question, k=1)[0][1]
_rel = vectorstore.similarity_search_with_relevance_scores(test_question, k=1)[0][1]
print(f"Coseno crudo: {_raw:.3f} | relevancia del retriever: {_rel:.3f} | (coseno+1)/2: {to_relevance(_raw):.3f}")

Coseno crudo: 0.571 | relevancia del retriever: 0.785 | (coseno+1)/2: 0.785


---
## Parte 5 — Quiz sobre los papers

Ejecuta cada pregunta y compara la respuesta del RAG con el *ground truth*. Las preguntas están en inglés para coincidir con el idioma de los artículos; el prompt pide responder en el mismo idioma de la consulta.

Además de la corrección factual, observa si el contexto recuperado proviene del paper adecuado.


In [8]:
# ── Pregunta 1 ────────────────────────────────────────────────
q1 = "What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?"

answer1 = rag_chain.invoke(q1)
print(f"Q: {q1}")
print(f"A: {answer1}")

Q: What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?
A: 28.4


<details>
<summary>✅ Ground Truth — Pregunta 1</summary>

El Transformer (big model) obtuvo un BLEU score de **28.4** en la tarea de
traducción WMT 2014 English-to-German, superando los mejores resultados
previos (incluyendo ensembles) por más de 2 puntos BLEU.
*(Fuente: Abstract y Section 6.1 — Attention Is All You Need)*
</details>

In [9]:
# ── Pregunta 2 ────────────────────────────────────────────────
q2 = "How does RAG combine parametric and non-parametric memory?"

answer2 = rag_chain.invoke(q2)
print(f"Q: {q2}")
print(f"A: {answer2}")

Q: How does RAG combine parametric and non-parametric memory?
A: RAG combines parametric and non-parametric memory by using a pre-trained seq2seq model as the parametric memory and a dense vector index of Wikipedia accessed via a pre-trained neural retriever as the non-parametric memory, trained end-to-end with a probabilistic model.


<details>
<summary>✅ Ground Truth — Pregunta 2</summary>

RAG combina **memoria paramétrica** (un modelo seq2seq pre-entrenado, BART-large)
con **memoria no-paramétrica** (un índice denso de vectores de Wikipedia).
El retriever (DPR) recupera los top-K documentos relevantes para la query,
y el generador (BART) los usa como contexto adicional para producir la respuesta.
Ambos componentes se entrenan end-to-end de forma conjunta.
*(Fuente: Abstract y Section 2 — RAG paper)*
</details>

In [10]:
# ── Pregunta 3 ────────────────────────────────────────────────
q3 = "What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?"

answer3 = rag_chain.invoke(q3)
print(f"Q: {q3}")
print(f"A: {answer3}")

Q: What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?
A: The scaling factor sqrt(dk) is used to counteract the effect of dot products growing large in magnitude for large values of dk, which pushes the softmax function into regions with extremely small gradients, making training difficult.


<details>
<summary>✅ Ground Truth — Pregunta 3</summary>

El factor de escala **1/√dk** se aplica para contrarrestar el efecto de que,
con valores grandes de dk, los productos punto crecen en magnitud y empujan
la función softmax hacia regiones con gradientes muy pequeños.
Esto haría el entrenamiento inestable. La división por √dk mantiene los
productos punto en un rango con gradientes útiles.
*(Fuente: Section 3.2.1 — Attention Is All You Need)*
</details>

In [11]:
# ── Pregunta 4 ────────────────────────────────────────────────
q4 = "What are the two RAG model variants proposed in the paper and how do they differ?"

answer4 = rag_chain.invoke(q4)
print(f"Q: {q4}")
print(f"A: {answer4}")

Q: What are the two RAG model variants proposed in the paper and how do they differ?
A: The two RAG model variants proposed in the paper are RAG-Sequence and RAG-Token. RAG-Sequence uses the same retrieved document to predict each target token, while RAG-Token can use different documents for different tokens.


<details>
<summary>✅ Ground Truth — Pregunta 4</summary>

Los dos variantes son:
- **RAG-Sequence**: usa el **mismo documento recuperado** para generar toda
  la secuencia de salida. Trata el documento como una variable latente única.
- **RAG-Token**: puede usar un **documento diferente para cada token** generado,
  permitiendo combinar información de múltiples fuentes en una sola respuesta.

RAG-Token es más flexible pero más costoso computacionalmente.
*(Fuente: Section 2.1 — RAG paper)*
</details>

In [12]:
# ── Pregunta 5 ────────────────────────────────────────────────
q5 = "How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?"

answer5 = rag_chain.invoke(q5)
print(f"Q: {q5}")
print(f"A: {answer5}")

Q: How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?
A: The base Transformer model uses 8 attention heads, with key and value dimensions of 64 per head.


<details>
<summary>✅ Ground Truth — Pregunta 5</summary>

El Transformer base usa **h = 8** attention heads en paralelo.
Para cada cabeza, las dimensiones son **dk = dv = dmodel/h = 512/8 = 64**.
El costo computacional total es similar al de single-head attention con
dimensionalidad completa, gracias a las dimensiones reducidas por cabeza.
*(Fuente: Section 3.2.2 — Attention Is All You Need)*
</details>

---
## Bonus — Pregunta entre ambos papers

Esta pregunta exige conectar información de los dos artículos. Además de generar la respuesta, inspeccionaremos los títulos y archivos de los fragmentos recuperados.


In [13]:
q_bonus = ("The RAG paper uses FAISS for fast retrieval with an HNSW approximation. "
           "The Attention paper proposes the Transformer architecture. "
           "What seq2seq model does RAG use as its generator component, "
           "and is it based on the Transformer?")

answer_bonus = rag_chain.invoke(q_bonus)

# Mostrar también qué chunks recuperó
docs_retrieved = retriever.invoke(q_bonus)
sources = [(d.metadata["title"], d.metadata["source"]) for d in docs_retrieved]

print(f"Q: {q_bonus}\n")
print(f"A: {answer_bonus}\n")
print(f"Fuentes recuperadas:")
for title, source in sources:
    print(f"  [{title}] — {source}")

Q: The RAG paper uses FAISS for fast retrieval with an HNSW approximation. The Attention paper proposes the Transformer architecture. What seq2seq model does RAG use as its generator component, and is it based on the Transformer?

A: The RAG paper uses BART as its seq2seq generator, and BART is based on the Transformer architecture.

Fuentes recuperadas:
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Retrieval-Augmented Generation] — rag_paper.pdf
  [Attention Is All You Need] — attention_paper.pdf


<details>
<summary>✅ Ground Truth — Bonus</summary>

RAG usa **BART-large** como generador (Section 2.3 del RAG paper).
BART es un modelo seq2seq pre-entrenado basado en la arquitectura **Transformer**
(referencia [58] en el RAG paper = el paper "Attention Is All You Need").
El RAG paper cita explícitamente: "BART-large, a pre-trained seq2seq transformer".

Esta pregunta demuestra el valor de RAG: el sistema conecta información de
dos documentos distintos para construir una respuesta completa.
</details>

---
# Tareas del workshop

## B1 · Evaluación contra el ground truth

Reutiliza las respuestas ya generadas arriba (`answer1` … `answer5`, `answer_bonus`), así que no hace llamadas nuevas al LLM. Columnas: chequeo automático por palabras clave del ground truth (lo confirmo a mano) y si el chunk top-1 vino del paper esperado.

In [14]:
ATT, RAGP = "attention_paper.pdf", "rag_paper.pdf"
QUIZ = {
    "Q1": dict(q=q1, a=answer1, src={ATT}, kw=[["28.4"]],
               resumen="BLEU del Transformer big en WMT14 En→De"),
    "Q2": dict(q=q2, a=answer2, src={RAGP}, kw=[["parametric"], ["non-parametric", "nonparametric"], ["BART"], ["DPR", "retriev", "Wikipedia"]],
               resumen="Memoria paramétrica y no paramétrica en RAG"),
    "Q3": dict(q=q3, a=answer3, src={ATT}, kw=[["softmax"], ["gradient"], ["large", "grow", "magnitude"]],
               resumen="Propósito de 1/√dk"),
    "Q4": dict(q=q4, a=answer4, src={RAGP}, kw=[["RAG-Sequence", "RAG Sequence"], ["RAG-Token", "RAG Token"], ["same", "single"], ["each token", "per token", "different"]],
               resumen="RAG-Sequence vs RAG-Token"),
    "Q5": dict(q=q5, a=answer5, src={ATT}, kw=[["8"], ["64"]],
               resumen="Cabezas y dk/dv del Transformer base"),
    "Bonus": dict(q=q_bonus, a=answer_bonus, src={ATT, RAGP}, kw=[["BART"], ["transformer"]],
                  resumen="Generador de RAG y relación con el Transformer"),
}

print("| # | Pregunta (resumen) | Respuesta del RAG (resumen) | Correcta (auto) | Paper correcto | Chunks recuperados (paper#chunk, coseno) |")
print("|---|---|---|---|---|---|")
for key, item in QUIZ.items():
    items = retrieved(vectorstore, item["q"])
    got = {src for _, src, _, _ in items}
    ok_src = "sí" if item["src"] <= got and items[0][1] in item["src"] else ("parcial" if item["src"] & got else "no")
    print(f"| {key} | {item['resumen']} | {cell(item['a'], 220)} | {grade(item['a'], item['kw'])} | {ok_src} | {fmt_retrieved(items)} |")

print("\nBonus — fuentes del retriever (umbral incluido):")
for d in retriever.invoke(q_bonus):
    print(f"  [{d.metadata['title']}] — {d.metadata['source']} (chunk {d.metadata['chunk_index']})")
print("¿Aparecieron los dos papers?", {d.metadata["source"] for d in retriever.invoke(q_bonus)} == {ATT, RAGP})

| # | Pregunta (resumen) | Respuesta del RAG (resumen) | Correcta (auto) | Paper correcto | Chunks recuperados (paper#chunk, coseno) |
|---|---|---|---|---|---|
| Q1 | BLEU del Transformer big en WMT14 En→De | 28.4 | sí | sí | attention#28 (0.71); attention#1 (0.66); attention#26 (0.58); attention#40 (0.52); attention#31 (0.50) |
| Q2 | Memoria paramétrica y no paramétrica en RAG | RAG combines parametric and non-parametric memory by using a pre-trained seq2seq model as the parametric memory and a dense vector index of Wikipedia accessed via a pre-trained neural retriever as the non-parametric mem… | parcial | sí | rag#1 (0.59); rag#8 (0.55); rag#32 (0.52); rag#7 (0.51); rag#50 (0.48) |
| Q3 | Propósito de 1/√dk | The scaling factor sqrt(dk) is used to counteract the effect of dot products growing large in magnitude for large values of dk, which pushes the softmax function into regions with extremely small gradients, making train… | sí | sí | attention#11 (0.55); attention#12 (0.48); a

## B2 · Experimentos con `chunk_size`, `k` y `score_threshold`

Se cambia **un parámetro a la vez**. Solo hay 3 construcciones de índice (400, 900, 1500 con overlap ≈ 13 %); los experimentos de `k` y umbral reutilizan el índice de 900. Las configuraciones repetidas (900/5/0.25) se cachean para no gastar llamadas.

Además de los umbrales pedidos (0.15, 0.25, 0.40, en la escala de *relevancia* del retriever), agrego un experimento **3b** con umbrales en coseno real (0.15, 0.25, 0.40 → relevancia 0.575, 0.625, 0.70) para ver el efecto que el enunciado espera.

In [15]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CORPUS = [
    (attention_text, "Attention Is All You Need", "attention_paper.pdf"),
    (rag_text, "Retrieval-Augmented Generation", "rag_paper.pdf"),
]

def build_store(chunk_size: int, chunk_overlap: int):
    """Chunking + embeddings + Qdrant en memoria. Devuelve (store, n_chunks)."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "],
    )
    docs = []
    for text, title, source in CORPUS:
        for i, chunk in enumerate(splitter.split_text(text)):
            if len(chunk.strip()) >= 30:
                docs.append(Document(page_content=chunk,
                                     metadata={"title": title, "source": source, "chunk_index": i}))
    name = f"exp_{chunk_size}_{chunk_overlap}"
    client = QdrantClient(":memory:")
    client.create_collection(collection_name=name,
                             vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
    store = QdrantVectorStore(client=client, collection_name=name, embedding=embeddings)
    with_retry(store.add_documents, docs)
    return store, len(docs)


def build_chain(store, k: int = 5, threshold: float = 0.25, chain_prompt=None):
    """Reutiliza un índice ya construido y cambia solo la recuperación."""
    retr = store.as_retriever(search_type="similarity_score_threshold",
                              search_kwargs={"k": k, "score_threshold": threshold})
    chain = ({"context": retr | format_docs, "question": RunnablePassthrough()}
             | (chain_prompt or prompt) | llm | StrOutputParser())
    return chain, retr


QIDS = ["Q1", "Q2", "Q3", "Q4", "Q5"]
STORES = {}
for cs, ov in [(400, 52), (900, 120), (1500, 195)]:
    STORES[cs] = build_store(cs, ov)
    print(f"✅ chunk_size={cs}, overlap={ov} → {STORES[cs][1]} chunks")

✅ chunk_size=400, overlap=52 → 326 chunks
✅ chunk_size=900, overlap=120 → 150 chunks
✅ chunk_size=1500, overlap=195 → 96 chunks


In [16]:
CACHE = {}

def run_config(cs, k, thr):
    key = (cs, k, round(thr, 3))
    if key in CACHE:
        return CACHE[key]
    store, n = STORES[cs]
    chain, retr = build_chain(store, k=k, threshold=thr)
    grades, n_ret, notes = [], [], []
    for qid in QIDS:
        q = QUIZ[qid]
        docs = retr.invoke(q["q"])
        ans = with_retry(chain.invoke, q["q"]) if docs else "(recuperación vacía) " + with_retry(chain.invoke, q["q"])
        g = grade(ans, q["kw"])
        grades.append(g); n_ret.append(len(docs))
        if g != "sí":
            notes.append(f"{qid}:{g}")
    res = dict(n=n, hits=grades.count("sí"), partial=grades.count("parcial"),
               avg_ret=sum(n_ret) / len(n_ret), notes=", ".join(notes) or "todas correctas")
    CACHE[key] = res
    return res

EXPERIMENTS = (
    [("1 · chunk_size", cs, 5, 0.25) for cs in (400, 900, 1500)]
    + [("2 · k", 900, k, 0.25) for k in (3, 5, 8)]
    + [("3 · threshold (relevancia)", 900, 5, t) for t in (0.15, 0.25, 0.40)]
    + [("3b · threshold (coseno real)", 900, 5, to_relevance(c)) for c in (0.15, 0.25, 0.40)]
)

print("| Exp. | chunk_size | k | score_threshold | # chunks índice | Chunks recuperados (prom.) | Aciertos (de 5) | Parciales | Observación |")
print("|---|---:|---:|---:|---:|---:|---:|---:|---|")
for exp, cs, k, thr in EXPERIMENTS:
    r = run_config(cs, k, thr)
    print(f"| {exp} | {cs} | {k} | {thr:.3f} | {r['n']} | {r['avg_ret']:.1f} | {r['hits']} | {r['partial']} | {r['notes']} |")

| Exp. | chunk_size | k | score_threshold | # chunks índice | Chunks recuperados (prom.) | Aciertos (de 5) | Parciales | Observación |
|---|---:|---:|---:|---:|---:|---:|---:|---|
| 1 · chunk_size | 400 | 5 | 0.250 | 326 | 5.0 | 4 | 1 | Q2:parcial |
| 1 · chunk_size | 900 | 5 | 0.250 | 150 | 5.0 | 4 | 1 | Q2:parcial |
| 1 · chunk_size | 1500 | 5 | 0.250 | 96 | 5.0 | 5 | 0 | todas correctas |
| 2 · k | 900 | 3 | 0.250 | 150 | 3.0 | 4 | 1 | Q2:parcial |
| 2 · k | 900 | 5 | 0.250 | 150 | 5.0 | 4 | 1 | Q2:parcial |
| 2 · k | 900 | 8 | 0.250 | 150 | 8.0 | 4 | 1 | Q2:parcial |
| 3 · threshold (relevancia) | 900 | 5 | 0.150 | 150 | 5.0 | 4 | 1 | Q2:parcial |
| 3 · threshold (relevancia) | 900 | 5 | 0.250 | 150 | 5.0 | 4 | 1 | Q2:parcial |
| 3 · threshold (relevancia) | 900 | 5 | 0.400 | 150 | 5.0 | 4 | 1 | Q2:parcial |
| 3b · threshold (coseno real) | 900 | 5 | 0.575 | 150 | 5.0 | 4 | 1 | Q2:parcial |
| 3b · threshold (coseno real) | 900 | 5 | 0.625 | 150 | 5.0 | 4 | 1 | Q2:parcial |
| 3b · t

## B3 · Preguntas trampa: ¿el RAG admite que no sabe?

Tres preguntas **plausibles** cuya respuesta no está en los papers (todas mencionan modelos o técnicas posteriores a 2020). Se imprime lo que recuperó el retriever para entender de dónde saldría una posible invención.

Prompt v2 (más estricto) para repetir:

```diff
- "Answer using only the supplied academic-paper excerpts. "
- "If the excerpts do not contain enough evidence, say so explicitly. "
+ "Answer using only the supplied academic-paper excerpts; never use outside knowledge. "
+ "Before answering, check that every model, dataset, number and name in the question "
+ "appears literally in the excerpts. If any of them is missing, reply exactly: "
+ "'The provided papers do not contain this information.' and nothing else. "
  "Answer in the same language as the question and be concise. "
  "Mention the paper title supporting the answer."
```

In [17]:
TRAPS = [
    "What BLEU score does GPT-4 achieve on WMT 2014 English-to-German according to the paper?",
    "By how many exact-match points does RAG outperform Llama 2 70B on Natural Questions?",
    "Which FlashAttention version does the Transformer use to reduce the memory cost of attention?",
]

prompt_v2 = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer using only the supplied academic-paper excerpts; never use outside knowledge. "
        "Before answering, check that every model, dataset, number and name in the question "
        "appears literally in the excerpts. If any of them is missing, reply exactly: "
        "'The provided papers do not contain this information.' and nothing else. "
        "Answer in the same language as the question and be concise. "
        "Mention the paper title supporting the answer.",
    ),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])
rag_chain_v2 = ({"context": retriever | format_docs, "question": RunnablePassthrough()}
                | prompt_v2 | llm | StrOutputParser())

def classify(answer):
    return "admite que no sabe" if is_refusal(answer) else "revisar: posible invención / mezcla"

print("| # | Pregunta trampa | Chunks que llegaron al LLM | Respuesta prompt v1 | Clasificación v1 (auto) | Respuesta prompt v2 | Clasificación v2 (auto) |")
print("|---|---|---|---|---|---|---|")
for i, q in enumerate(TRAPS, 1):
    a1 = with_retry(rag_chain.invoke, q)
    a2 = with_retry(rag_chain_v2.invoke, q)
    print(f"| T{i} | {cell(q, 90)} | {fmt_retrieved(retrieved(vectorstore, q))} | {cell(a1)} | {classify(a1)} | {cell(a2)} | {classify(a2)} |")

| # | Pregunta trampa | Chunks que llegaron al LLM | Respuesta prompt v1 | Clasificación v1 (auto) | Respuesta prompt v2 | Clasificación v2 (auto) |
|---|---|---|---|---|---|---|
| T1 | What BLEU score does GPT-4 achieve on WMT 2014 English-to-German according to the paper? | attention#28 (0.56); attention#1 (0.55); attention#26 (0.49); attention#24 (0.42); attention#40 (0.41) | 28.4 | revisar: posible invención / mezcla | The provided papers do not contain this information. | admite que no sabe |
| T2 | By how many exact-match points does RAG outperform Llama 2 70B on Natural Questions? | rag#29 (0.54); rag#30 (0.50); rag#25 (0.48); rag#9 (0.47); rag#42 (0.46) | 11.8 | revisar: posible invención / mezcla | 25.0 | revisar: posible invención / mezcla |
| T3 | Which FlashAttention version does the Transformer use to reduce the memory cost of attent… | attention#5 (0.45); attention#6 (0.42); attention#7 (0.40); attention#15 (0.40); attention#1 (0.39) | The Transformer does not use FlashAt

## B4 · Preguntas en español sobre papers en inglés

Se comparan Q1, Q3 y Q4 en inglés y en español: solapamiento de los top-5 chunks, coseno medio y calidad de la respuesta (chequeo automático con las mismas palabras clave). Para la hipótesis, también mido directamente el **coseno entre el embedding de la pregunta EN y el de la ES**, comparado con el de una pregunta no relacionada.

In [18]:
import numpy as np

ES = {
    "Q1": "¿Qué puntaje BLEU obtuvo el Transformer (modelo big) en WMT 2014 inglés-alemán?",
    "Q3": "¿Cuál es el propósito del factor de escala sqrt(dk) en la atención de producto punto escalado?",
    "Q4": "¿Cuáles son las dos variantes del modelo RAG propuestas en el paper y en qué se diferencian?",
}

print("| # | Overlap top-5 (EN∩ES) | Coseno medio EN | Coseno medio ES | Chunks EN | Chunks ES | Correcta EN | Correcta ES | Respuesta ES (resumen) |")
print("|---|---:|---:|---:|---|---|---|---|---|")
for qid, q_es in ES.items():
    q_en = QUIZ[qid]["q"]
    r_en, r_es = retrieved(vectorstore, q_en), retrieved(vectorstore, q_es)
    ids_en = {(s, i) for _, s, i, _ in r_en}; ids_es = {(s, i) for _, s, i, _ in r_es}
    a_es = with_retry(rag_chain.invoke, q_es)
    print(f"| {qid} | {len(ids_en & ids_es)}/5 | {np.mean([x[3] for x in r_en]):.3f} | {np.mean([x[3] for x in r_es]):.3f} "
          f"| {fmt_retrieved(r_en)} | {fmt_retrieved(r_es)} | {grade(QUIZ[qid]['a'], QUIZ[qid]['kw'])} | {grade(a_es, QUIZ[qid]['kw'])} | {cell(a_es, 160)} |")

# Prueba de la hipótesis: ¿el modelo de embeddings alinea idiomas?
def cos(a, b):
    a, b = np.array(a), np.array(b); return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
unrelated = "¿Cuál es la receta tradicional de la bandeja paisa?"
print("\n| # | coseno(EN, ES) | coseno(EN, no relacionada) |\n|---|---:|---:|")
for qid, q_es in ES.items():
    e_en, e_es, e_un = (embeddings.embed_query(t) for t in (QUIZ[qid]["q"], q_es, unrelated))
    print(f"| {qid} | {cos(e_en, e_es):.3f} | {cos(e_en, e_un):.3f} |")

| # | Overlap top-5 (EN∩ES) | Coseno medio EN | Coseno medio ES | Chunks EN | Chunks ES | Correcta EN | Correcta ES | Respuesta ES (resumen) |
|---|---:|---:|---:|---|---|---|---|---|
| Q1 | 4/5 | 0.592 | 0.584 | attention#28 (0.71); attention#1 (0.66); attention#26 (0.58); attention#40 (0.52); attention#31 (0.50) | attention#28 (0.71); attention#1 (0.64); attention#26 (0.58); attention#40 (0.50); attention#27 (0.49) | sí | sí | 28.4 |
| Q3 | 4/5 | 0.398 | 0.270 | attention#11 (0.55); attention#12 (0.48); attention#13 (0.38); attention#16 (0.30); attention#20 (0.28) | attention#11 (0.35); attention#12 (0.33); attention#13 (0.30); attention#16 (0.20); attention#2 (0.18) | sí | parcial | El propósito del factor de escala sqrt(dk) es contrarrestar el efecto de que los productos punto crezcan en magnitud para valores grandes de dk, lo que empuja … |
| Q4 | 5/5 | 0.474 | 0.453 | rag#10 (0.51); rag#11 (0.47); rag#1 (0.47); rag#50 (0.46); rag#7 (0.45) | rag#10 (0.49); rag#11 (0.46); rag#50 (0

## B5 · Mi propio quiz con ground truth

Tres preguntas nuevas. Verifiqué cada respuesta leyendo el texto de los PDFs. La tercera exige **combinar** los dos papers.

In [19]:
# ── Pregunta B5-1 ────────────────────────────────────────────
b5_1 = ("Which optimizer and hyperparameters (beta1, beta2, epsilon) were used to train the Transformer, "
        "and how many warmup steps does its learning-rate schedule use?")
answer_b5_1 = with_retry(rag_chain.invoke, b5_1)
print(f"Q: {b5_1}\nA: {answer_b5_1}")

Q: Which optimizer and hyperparameters (beta1, beta2, epsilon) were used to train the Transformer, and how many warmup steps does its learning-rate schedule use?
A: The Transformer used the Adam optimizer with β₁ = 0.9, β₂ = 0.98, and ε = 10⁻⁹, and its learning-rate schedule uses warmup_steps = 4000.


<details>
<summary>✅ Ground Truth — B5-1</summary>

Se usó **Adam** con **β1 = 0.9, β2 = 0.98 y ε = 10⁻⁹**. La tasa de aprendizaje sigue
`lrate = d_model^-0.5 · min(step^-0.5, step · warmup_steps^-1.5)`: crece linealmente durante
los primeros **warmup_steps = 4000** pasos y luego decae proporcional a 1/√step.
*(Fuente: Section 5.3 Optimizer — Attention Is All You Need)*
</details>

In [20]:
# ── Pregunta B5-2 ────────────────────────────────────────────
b5_2 = ("What Wikipedia dump does RAG use as its non-parametric knowledge source, how is each article split, "
        "and how many documents does the resulting index contain?")
answer_b5_2 = with_retry(rag_chain.invoke, b5_2)
print(f"Q: {b5_2}\nA: {answer_b5_2}")

Q: What Wikipedia dump does RAG use as its non-parametric knowledge source, how is each article split, and how many documents does the resulting index contain?
A: RAG uses the December 2018 Wikipedia dump, splits each article into disjoint 100-word chunks, and builds a total of 21M documents.


<details>
<summary>✅ Ground Truth — B5-2</summary>

RAG usa el dump de Wikipedia de **diciembre de 2018**. Cada artículo se divide en
**fragmentos disjuntos de 100 palabras**, para un total de **21 millones de documentos**.
Se indexan con un índice MIPS de **FAISS** con aproximación **HNSW**.
*(Fuente: Section 3 Experiments — RAG paper)*
</details>

In [21]:
# ── Pregunta B5-3 (combina los dos papers) ──────────────────────
b5_3 = ("How many parameters does the big Transformer model have, and how many trainable parameters "
        "do the RAG models have? Which one is larger?")
answer_b5_3 = with_retry(rag_chain.invoke, b5_3)
print(f"Q: {b5_3}\nA: {answer_b5_3}")
print("Fuentes:", [(d.metadata["source"], d.metadata["chunk_index"]) for d in retriever.invoke(b5_3)])

⏳ Rate limit; reintento en 20s
Q: How many parameters does the big Transformer model have, and how many trainable parameters do the RAG models have? Which one is larger?
A: The big Transformer model has 11 billion trainable parameters, while the RAG models have 626 million trainable parameters. The big Transformer model is larger.
Fuentes: [('rag_paper.pdf', 91), ('rag_paper.pdf', 10), ('rag_paper.pdf', 93), ('rag_paper.pdf', 1), ('rag_paper.pdf', 7)]


<details>
<summary>✅ Ground Truth — B5-3</summary>

El Transformer **big** tiene **213 millones** de parámetros (Table 3 — Attention Is All You Need).
Los modelos RAG tienen **626 millones de parámetros entrenables**: 110M del codificador de consultas
BERT-base de DPR, 110M del codificador de documentos (que no se entrena) y 406M de BART-large
(Appendix G — RAG paper). RAG es unas **2,9 veces** más grande en parámetros entrenables, aunque mucho
más pequeño que T5-11B, con el que se compara.
</details>

In [22]:
B5 = {
    "B5-1": dict(q=b5_1, a=answer_b5_1, src={ATT}, kw=[["Adam"], ["0.98"], ["4000", "4,000"]]),
    "B5-2": dict(q=b5_2, a=answer_b5_2, src={RAGP}, kw=[["2018"], ["100"], ["21"]]),
    "B5-3": dict(q=b5_3, a=answer_b5_3, src={ATT, RAGP}, kw=[["213"], ["626"]]),
}
print("| # | Pregunta | Respuesta del RAG (resumen) | Correcta (auto) | Papers en top-5 | Chunks (paper#chunk, coseno) |")
print("|---|---|---|---|---|---|")
for key, it in B5.items():
    items = retrieved(vectorstore, it["q"])
    got = {s for _, s, _, _ in items}
    print(f"| {key} | {cell(it['q'], 90)} | {cell(it['a'])} | {grade(it['a'], it['kw'])} | "
          f"{'ambos' if got == {ATT, RAGP} else ', '.join(sorted(got))} | {fmt_retrieved(items)} |")

| # | Pregunta | Respuesta del RAG (resumen) | Correcta (auto) | Papers en top-5 | Chunks (paper#chunk, coseno) |
|---|---|---|---|---|---|
| B5-1 | Which optimizer and hyperparameters (beta1, beta2, epsilon) were used to train the Transf… | The Transformer used the Adam optimizer with β₁ = 0.9, β₂ = 0.98, and ε = 10⁻⁹, and its learning-rate schedule uses warmup_steps = 4000. | sí | attention_paper.pdf | attention#25 (0.52); attention#29 (0.45); attention#39 (0.45); attention#1 (0.44); attention#27 (0.42) |
| B5-2 | What Wikipedia dump does RAG use as its non-parametric knowledge source, how is each arti… | RAG uses the December 2018 Wikipedia dump, splits each article into disjoint 100-word chunks, and builds a total of 21M documents. | sí | rag_paper.pdf | rag#18 (0.51); rag#1 (0.50); rag#7 (0.47); rag#42 (0.45); rag#37 (0.44) |
| B5-3 | How many parameters does the big Transformer model have, and how many trainable parameter… | The big Transformer model has 11 billion trainable para

---
## Reflexión final

1. **¿Qué ocurrió cuando el contexto no contenía información suficiente?** ¿El modelo reconoció la limitación o inventó una respuesta?
2. **¿Qué fragmentos recuperó para la pregunta bonus?** ¿Aparecieron ambos papers?
3. **¿Qué efecto tendría modificar `chunk_size`, `chunk_overlap`, `k` o `score_threshold`?**
4. **¿Por qué conviene separar el modelo de embeddings del modelo generativo?**
5. **¿Qué cambiarías para llevar este prototipo a producción?** Considera persistencia, control de acceso, evaluación, observabilidad y costos.


> Mis respuestas (R1), apoyadas en las tablas de B1–B5, están en `RESPUESTAS.md`, sección «Reflexión».